In [5]:
from pyope import *
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [6]:
clear_registry()
b = BasicOperator('b', conformal_weight=3/2, fermionic=True)
c = BasicOperator('c', conformal_weight=-1/2, fermionic=True)
β = BasicOperator('β', conformal_weight=1)
γ = BasicOperator('γ', conformal_weight=0)

OPE[b, c] = MakeOPE([One])
OPE[β, γ] = MakeOPE([-One])

In [7]:
Jplus = β
J0 = NO(b,c) + 2 * NO(β,γ)
Jminus = - NO(β, NO(γ, γ)) - NO(γ, NO(b, c)) + (3/2) * d(γ)
Gplus = b
Gminus = NO(b, γ)
Gtildeplus = NO(c, d(β)) + 2 * NO(d(c), β)
Gtildeminus = -NO(b, NO(d(c), c)) + 2 * NO(β, NO(γ, d(c))) + NO(d(β), NO(γ, c)) - 3/2 * d(d(c)) 
T = - 3/2 * NO(b, d(c)) - NO(β, d(γ)) - 1/2 * NO(d(b), c)

generators = [Jplus, J0, Jminus, Gplus, Gminus, Gtildeplus, Gtildeminus, T]

(Jplus, J0, Jminus, Gplus,
 Gminus, Gtildeplus, Gtildeminus, T) = make_realized(generators)
generators = [Jplus, J0, Jminus, Gplus, Gminus, Gtildeplus, Gtildeminus, T]

# Check Jacobi

In [10]:
[check_jacobi_identity(g1, g2, g3) for g1 in generators for g2 in generators for g3 in generators]

KeyboardInterrupt: 

In [14]:
def flatten_and_deduplicate(lst):
    def flatten_completely(lst):
        result = []
        for item in lst:
            if isinstance(item, list):
                result.extend(flatten_completely(item))
            else:
                result.append(item)
        return result
    flatten = flatten_completely(lst)

    unique = []
    for item in flatten:
        if item not in unique:
            unique.append(item)
    return unique

In [12]:
[check_jacobi_identity(T, g2, g3)
 for g2 in generators for g3 in generators]

[[],
 [[ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')]],
 [[ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')]],
 [],
 [[ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')]],
 [],
 [[ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')]],
 [[ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero')],
  [ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero')],
  [ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero'),
   ConstantOperator('Zero')]],
 [[ConstantOperator('Zero'), ConstantOperator('Zero')],
  [ConstantOperator('Zero'), ConstantOperator('Zero')]],
 [[ConstantO

In [15]:
jacobi_identities = [check_jacobi_identity(T, g2, g3)
for g2 in generators for g3 in generators]


non_zero = flatten_and_deduplicate(jacobi_identities)
[simplify(expr) for expr in non_zero]

[ConstantOperator('Zero')]

In [16]:
non_zero = flatten_and_deduplicate(Out[11])
[simplify(expr) for expr in non_zero]

[ConstantOperator('Zero')]

# Null state

In [9]:
from sympy import symbols
a1, a2, a3 = symbols('a1 a2 a3')

# 用 generators 所生成的算符空间
basis = LocalOperatorBasis(generators, max_weight=4)

# 用 b, c, β, γ 所生成的算符空间
bc_basis = LocalOperatorBasis([b, c, β, γ], max_weight=4, max_occurence=6)

# 找到 Sugawara relation


exp = simplify((T - a1 * NO(Jplus, Jminus) + a2 *
               NO(J0, J0) + a3 * NO(Jminus, Jplus)).realize())

list_of_ops = [T, NO(Jplus, Jminus), NO(J0, J0), NO(Jminus, Jplus)]
list_of_ops_realized = [T.realize(), NO(Jplus, Jminus).realize(), NO(J0, J0).realize(), NO(Jminus, Jplus).realize()]

coefficients = bc_basis.list_zero_relations(list_of_ops_realized, weight=2)[0]["coefficients"]

sum([coefficients[i] * list_of_ops[i] for i in range(len(list_of_ops))])

0.5*NO(J0,J0) + NO(Jminus,Jplus) + 1.0*NO(Jplus,Jminus) - 1.0*T

In [10]:
# 用 generators 所生成的算符空间
basis = LocalOperatorBasis(generators, max_weight=4)

# 用 b, c, β, γ 所生成的算符空间
bc_basis = LocalOperatorBasis([b, c, β, γ], max_weight=4, max_occurence=6)

# 列举 weight=2 的算符，有 10 个
list_of_ops = [op for op in basis.list(weight=2)]
list_of_ops_realized = [op.realize() for op in basis.list(weight=2)]
len(list_of_ops)

# 检查 那 10 个算符是否独立，只有 9 个独立：有 1 个 null state
# bc_basis.list_independent_ops(list_of_ops, weight=2)

coefficients = bc_basis.list_zero_relations(list_of_ops_realized, weight=2)[0]["coefficients"]
sum([coefficients[i] * list_of_ops[i] for i in range(len(list_of_ops_realized))])

10

-1.0*∂J0 - 0.5*NO(J0,J0) - 2.0*NO(Jminus,Jplus) + T

In [33]:
# 用 generators 所生成的算符空间
basis = LocalOperatorBasis(generators, max_weight=4)

# 用 b, c, β, γ 所生成的算符空间
bc_basis = LocalOperatorBasis([b, c, β, γ], max_weight=4, max_occurence=10)

# 列举 weight=2 的算符，有 10 个
list_of_ops = [op for op in basis.list(weight=5/2)]
list_of_ops_realized = [simplify(op.realize()) for op in basis.list(weight=5/2)]
len(list_of_ops)

# 检查 那 10 个算符是否独立，只有 9 个独立：有 1 个 null state
# bc_basis.list_independent_ops(list_of_ops, weight=4)

relations = bc_basis.list_zero_relations(
    list_of_ops_realized, weight=5/2)
len(relations)

16

4

In [36]:
sum([relations[0]["coefficients"][i] * list_of_ops[i]
    for i in range(len(list_of_ops_realized))])


sum([relations[1]["coefficients"][i] * list_of_ops[i] for i in range(len(list_of_ops_realized))])

sum([relations[2]["coefficients"][i] * list_of_ops[i]
    for i in range(len(list_of_ops_realized))])


sum([relations[3]["coefficients"][i] * list_of_ops[i]
    for i in range(len(list_of_ops_realized))])

2*∂Gplus - 2*NO(Gminus,Jplus) + NO(Gplus,J0)

-1.0*∂Gminus + NO(Gminus,J0)/2 + NO(Gplus,Jminus)

2*∂Gtildeplus - 2*NO(Gtildeminus,Jplus) + NO(Gtildeplus,J0)

-1.0*∂Gtildeminus + 0.5*NO(Gtildeminus,J0) + NO(Gtildeplus,Jminus)

In [26]:
sum([coefficients[i] * list_of_ops[i]
    for i in range(len(list_of_ops_realized))])

NO(Gminus,NO(Gplus,J0))/2 + NO(Gminus,∂Gplus)

In [24]:
NO(b, NO(d(c),NO(d(c,2),NO(c,NO(β,γ)))))

NO(b, NO(∂c,NO(∂^2c,NO(c,NO(β,γ)))))